## Multi-Turn RL with SageMaker AI, part 2: train with `MultiTurnRLTrainer`

You turn the task into a prompt dataset, launch a GRPO training job against the agent you deployed
in part 1, and learn to read a run in MLflow - metrics *and* trajectories.

| # | Notebook | What you do |
|---|---|---|
| 1 | `01_build_and_deploy_agent.ipynb` | Explore the task, tools, and reward; deploy the rollout agent to Amazon Bedrock AgentCore |
| 2 | `02_train.ipynb` | Build the prompt dataset and launch a GRPO training job with `MultiTurnRLTrainer`; read MLflow |
| 3 | `03_evaluate_and_deploy.ipynb` | Compare base vs. fine-tuned with `MultiTurnRLEvaluator`, diagnose failures, deploy, clean up |

**Before you start:** complete `01_build_and_deploy_agent.ipynb` (prerequisites stack deployed,
dependencies installed, agent `READY`).

### 1. Configuration

In [ ]:
import os
import sys
from pathlib import Path

import boto3
from sagemaker.train.multi_turn_rl_trainer import MultiTurnRLTrainer

STACK_NAME = "mtrl-prerequisites"  # the prerequisites.yaml stack you deployed
AGENT_NAME = "sopbench_agentcore"  # the name passed to `agentcore configure --name`
PREFIX = "sopbench-mtrl/aircraft"  # S3 prefix for the prompt parquets
AGENT_DIR = Path("sopbench_agent").resolve()

# Make `!python` / `!agentcore` resolve to this kernel's environment.
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"

sess = boto3.session.Session()
REGION = sess.region_name
assert REGION, "Set AWS_REGION (or a default region) to the region you deployed the stack in."
# The training job must run in the AgentCore runtime's region: pin it for every SDK client.
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"] = REGION
account = sess.client("sts").get_caller_identity()["Account"]

stack = sess.client("cloudformation").describe_stacks(StackName=STACK_NAME)["Stacks"][0]
outs = {o["OutputKey"]: o["OutputValue"] for o in stack.get("Outputs", [])}
BUCKET = outs["PromptsBucketName"]
ROLE_ARN = outs["JobRoleArn"]
AGENTCORE_EXEC_ROLE_ARN = outs["AgentCoreExecRoleArn"]
MLFLOW_APP_ARN = outs["MlflowAppArn"]
TRAIN_DATASET = f"s3://{BUCKET}/{PREFIX}/train/aircraft_inspection_train.parquet"
VAL_DATASET = f"s3://{BUCKET}/{PREFIX}/validation/aircraft_inspection_val.parquet"


def find_agent_runtime_arn(name=AGENT_NAME):
    """ARN of the READY AgentCore runtime with exactly this name (others may share a prefix)."""
    for runtime in MultiTurnRLTrainer.list_bedrock_agentcore_runtimes(session=sess):
        if runtime["name"] == name and runtime["status"] == "READY":
            return runtime["arn"]
    return None


print(f"account={account} region={REGION} bucket={BUCKET}")
AGENT_RUNTIME_ARN = find_agent_runtime_arn()
S3_OUTPUT_PATH = f"s3://{BUCKET}/output/"
print("AGENT_RUNTIME_ARN =", AGENT_RUNTIME_ARN)
assert AGENT_RUNTIME_ARN, f"No READY runtime named {AGENT_NAME} - run 01_build_and_deploy_agent.ipynb."

### 2. Build and upload the prompt dataset

`scripts/preprocess.py` turns each SOP-Bench task into one parquet row with a single `prompt` column
- a JSON string with `benchmark`, `instance_id`, `task_input`, and `ground_truth`. The service
forwards that string verbatim to the agent, so this schema is the contract between the data and the
agent. It makes a deterministic 80/20 train/validation split (89 / 23 prompts).

In [ ]:
!cd {AGENT_DIR} && PYTHONPATH=src python scripts/preprocess.py --s3-prefix s3://{BUCKET}/{PREFIX}

### 3. How a training step works

Each step, the trainer samples a batch of prompts and asks the agent for **`group_size`** rollouts
per prompt. The reward scores each rollout; **GRPO** computes each rollout's advantage relative to
its group's mean and updates the policy (a LoRA adapter on the base model).

Because the advantage is *relative within the group*, a group whose rollouts all get the same
reward contributes **no learning signal** - which is why partial credit and reward variance matter.

### 4. Build the trainer

The fine-tuned adapter is registered in a model package group, auto-created as
`{model}-mtrl-mpg` (checkpoints in `{model}-mtrl-checkpoint-mpg`) and **shared by every MTRL job in
the account**. To keep your runs separate, pass `output_model_package_group=` and
`intermediate_checkpoint_model_package_group=` (two different groups).

In [ ]:
from sagemaker.train.multi_turn_rl_trainer import MultiTurnRLTrainer

MODEL = "openai-reasoning-gpt-oss-20b"
assert MODEL in MultiTurnRLTrainer.list_supported_models(), f"{MODEL} not supported"

trainer = MultiTurnRLTrainer(
    model=MODEL,
    agent_env=AGENT_RUNTIME_ARN,          # AgentCore runtime ARN (managed path)
    training_dataset=TRAIN_DATASET,
    validation_dataset=VAL_DATASET,
    mlflow_app_arn=MLFLOW_APP_ARN,
    s3_output_path=S3_OUTPUT_PATH,
    role=ROLE_ARN,
    accept_eula=True,                     # gated model
)
print(trainer.hyperparameters.get_info())  # authoritative names, types, defaults, and ranges

### 5. Hyperparameters

A ready-to-run recipe. A step takes about 3-4 minutes, so the 10 steps below finish in well under
an hour; a 3-step test run took about 12 minutes and raised held-out `val/reward/mean` from 2.4 to
4.7 (out of 5). With 89 prompts and `global_batch_size=32` an epoch is
only ~3 steps, so `max_epochs` is a high cap and **`max_steps` stops the run**. Lower `max_steps`
(e.g. 3) to finish faster; the flow is identical.

| Knob | Value | Why |
|---|---|---|
| `advantage_method` | `group_based` | GRPO; the library also offers per-turn and other estimators |
| `group_size` | 4 | rollouts per prompt; larger = better baseline, more rollouts per step |
| `global_batch_size` | 32 | prompts per step |
| `sampling_max_tokens` | 4096 | per-rollout token cap; also bounds trajectory length |
| `learning_rate`, `lora_rank`, `lora_alpha` | 4e-5, 32, 64 | LoRA adapter training |
| `val_every` | 10 | validation at step 0 and step 10 |
| `success_threshold` | 5.0 | pass@k counts a rollout as a success when reward >= this; 5.0 (`FINAL_REWARD_SCALE`) = every field correct. The default 1.0 would count ~20%-correct answers. |
| `rollout_max_concurrency` | 64 | the **only** throttle on AgentCore; lower toward 32 and/or raise `rollout_max_retries` if you see `httpx ConnectTimeout` |

Concurrency trades wall-clock for connection pressure; it does not change the reward the model
reaches. A longer recipe (`max_steps=30`, `global_batch_size=64`, `sampling_max_tokens=8192`) is in
`sopbench_agent/scripts/run_rft_training.py`, the non-interactive equivalent of this notebook. It
took about 2 h 15 min and raised held-out pass@1 from 0.48 to 0.87.

In [ ]:
from sopbench_agentcore.reward import FINAL_REWARD_SCALE

hp = trainer.hyperparameters
hp.advantage_method = "group_based"   # GRPO
hp.loss_fn = "ppo"
hp.global_batch_size = 32
hp.group_size = 4
hp.sampling_max_tokens = 4096
hp.learning_rate = 4e-5
hp.lora_rank = 32
hp.lora_alpha = 64
hp.max_epochs = 15                    # high cap; max_steps governs when the run stops
hp.max_steps = 10
hp.val_every = 10
hp.success_threshold = FINAL_REWARD_SCALE  # pass@k success = a fully correct answer (5.0)
hp.rollout_max_concurrency = 64
hp.rollout_max_retries = 3

### 6. Launch (non-blocking)

In [ ]:
job = trainer.train(wait=False)
print("Job:   ", job.job_name)
print("Status:", job.job_status)
print("MLflow:", job.get_mlflow_url())

In [ ]:
job.refresh()
print(f"status={job.job_status} secondary={getattr(job, 'secondary_status', '?')}")

### 7. Read the run in MLflow

MLflow is your primary observability surface: metrics stream every step, so you diagnose a run
*while it runs*. Open it from the URL above, or from the SageMaker console (MLflow -> your app ->
**Open MLflow**) - printed links carry a short-lived token. Your jobs appear in the **Runs** list:

![MLflow Runs list for the openai-reasoning-gpt-oss-20b experiment, one row per training job with status and a duration of roughly 2-3 hours.](images/mlflow-all-runs.png)

**Watch these first:**

| Metric | Read it as |
|---|---|
| `rollout/reward/mean`, `rollout/reward/valid_mean` | training reward (valid_mean: over non-zero-advantage groups) - should climb |
| `analysis/zero_adv_groups` | groups with no variance, so no learning signal - high means a saturated or too-coarse reward |
| `rollout/tokens/response_max` | if it pins at `sampling_max_tokens`, rollouts truncate mid-answer |
| `val/reward/mean` | held-out reward every `val_every` steps - should **track** training reward |

Switch to chart mode and MLflow groups related metrics, such as the `rollout/reward` group
(`max`, `mean`, `min`). Rewards lie in `[-1, 5]`.

Mean climbing and holding is the model learning the output contract and tool sequence; max reaching
5 means the best rollouts are already perfect; min lifting off the floor (and away from
the `-1` no-final-answer penalty) means even the worst rollouts improve. The min-max spread is what
GRPO learns from. If `val/reward/mean` **tracks** training reward, the model is generalizing; if it
flattens while training reward keeps rising, suspect a reward hack.

In-training validation is forced to greedy `temperature=0`, so its pass@k is degenerate - use the
standalone evaluator in part 3 for real pass@k.

### 8. Read the trajectories

Metrics say the run is healthy; **traces** say what the agent actually did. The **Traces** tab lists
one row per rollout - add the `reward` column from **Columns** for fast triage:

![MLflow Traces tab: one row per aircraft_inspection rollout with token counts around 22,000-28,000 and a reward column showing mostly 5 with one row at 4.16667.](images/mlflow-traces-all.png)

![MLflow Traces Columns menu with attributes such as Trace ID, Request, Response, Tokens, and Execution time.](images/mlflow-trace-columns.png)

Open a trace: **Summary** shows the exact task inputs and each turn's tool call in order;
**Details & Timeline** shows every turn's `thinking`, `tool_calls`, and tool result:

![MLflow trace Summary for aircraft a_00262: the prompt JSON followed by turn-1 VerifyAircraftClearance, turn-2 VerifyMechanicalComponents, turn-3 VerifyElectricalSystems, turn-4 CrossCheckSpecifications.](images/mlflow-trace-landing-page.png)

![MLflow trace Details and Timeline: span tree from turn-1 to turn-8 including the three reporting tools, and turn-1's thinking and tool_calls JSON.](images/mlflow-trace-details.png)

Read off a trace: the **tool sequence** (a skipped step or a loop on one tool explains a low
reward), **where the reward lands** (on the root `trajectory` span and the final turn; intermediate
turns carry 0), and **per-turn tokens** (which turn is long when `response_max` is high).

### 9. Reattach after a kernel restart

Find your job and reattach - no need to rerun the notebook.

In [ ]:
from itertools import islice

from sagemaker.train.agent_rft_job import AgentRFTJob

for j in islice(AgentRFTJob.get_all(sort_by="CreationTime", sort_order="Descending"), 20):
    print(f"{j.job_status:<12} {j.job_name}")

In [ ]:
# job = MultiTurnRLTrainer.attach(job_name="<JobName from the list above>")
# job.refresh(); print(job.job_status, job.get_mlflow_url())
# job.stop()   # stop a running job

When the job is `Completed`, the LoRA adapter is registered as a SageMaker Model Package
(`job.output_model_package_arn`). **Next:** `03_evaluate_and_deploy.ipynb`.